# FIT-VTON vs IDM-VTON: Comparison Demo

**Paper idea:** _Size-Aware Virtual Try-On — Conditioning IDM-VTON on Body-Garment Fit Measurements_

This notebook runs **fully in Colab with no large model downloads**. It:
1. Clones the repo and installs dependencies
2. Generates a synthetic FIT dataset (random realistic measurements + PIL images)
3. Shows the MeasurementEncoder producing distinct tokens per fit scenario
4. **Simulates** try-on results for Baseline (IDM-VTON, no fit conditioning) vs FIT-VTON (with adapter)
5. Visualises a full comparison grid across fit deltas
6. Plots simulated metric comparisons (FID / LPIPS / SSIM / Fit Accuracy)

> **Note:** Simulated results use geometric image transforms to visualise the expected behaviour of the fit adapter (tightness vs bagginess). Real results require the IDM-VTON download (~10 GB) — see `train.py` / `inference.py`.

In [ ]:
# ── Cell 1: Install & Clone ─────────────────────────────────────────────────
import subprocess, sys, os

REPO_URL = 'https://github.com/NatalieCarlebach1/fit-vton'
REPO_DIR = 'fit-vton'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned {REPO_URL}')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    print('Repo already present — pulled latest.')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install lightweight deps only (no torch reinstall if already present)
deps = [
    'diffusers==0.27.2',
    'transformers==4.38.2',
    'accelerate==0.27.2',
    'peft==0.9.0',
    'einops==0.7.0',
    'lpips==0.1.4',
    'clean-fid==0.1.35',
    'omegaconf==2.3.0',
    'rich==13.7.1',
    'seaborn==0.13.2',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'] + deps, check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Imports & Config ────────────────────────────────────────────────
import json, random, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont, ImageFilter

sns.set_theme(style='whitegrid', palette='muted')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SIZE_LABELS = ['XS', 'S', 'M', 'L', 'XL', 'XXL']
SIZE_TO_IDX = {s: i for i, s in enumerate(SIZE_LABELS)}

# Per-size approximate body stats (cm)
SIZE_STATS = {
    'XS': dict(chest=80, waist=62, hip=87),
    'S':  dict(chest=86, waist=68, hip=93),
    'M':  dict(chest=92, waist=74, hip=99),
    'L':  dict(chest=98, waist=80, hip=105),
    'XL': dict(chest=104, waist=86, hip=111),
    'XXL':dict(chest=110, waist=92, hip=117),
}

COLORS = {
    'IDM-VTON\n(baseline)': '#e07b54',
    'FIT-VTON\n(ours)':     '#4a90d9',
}
Path('outputs').mkdir(exist_ok=True)
print('Ready.')

In [ ]:
# ── Cell 3: Generate Synthetic FIT Dataset ──────────────────────────────────
import subprocess, sys

DATASET_DIR = 'data/fit_dataset_colab'

result = subprocess.run(
    [sys.executable, 'scripts/download_fit_dataset.py',
     '--output_dir', DATASET_DIR,
     '--split', 'all',
     '--mini'],
    capture_output=False
)

# Load metadata
with open(f'{DATASET_DIR}/train/metadata.json') as f:
    train_meta = json.load(f)
with open(f'{DATASET_DIR}/test/metadata.json') as f:
    test_meta = json.load(f)

print(f'Train: {len(train_meta)} samples | Test: {len(test_meta)} samples')
print('\nExample entry:')
import pprint; pprint.pprint(train_meta[0])

In [ ]:
# ── Cell 4: Dataset Statistics ──────────────────────────────────────────────
fit_deltas  = [m['fit_delta']        for m in train_meta]
body_idxs   = [m['body_size_idx']    for m in train_meta]
garment_idxs= [m['garment_size_idx'] for m in train_meta]
heights     = [m['height_cm']        for m in train_meta]
weights     = [m['weight_kg']        for m in train_meta]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Synthetic FIT Dataset Statistics', fontsize=14, fontweight='bold')

# 1. Fit delta
from collections import Counter
delta_counts = Counter(fit_deltas)
xs = sorted(delta_counts)
axes[0].bar(xs, [delta_counts[x] for x in xs], color='steelblue', edgecolor='white', width=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Fit Delta (garment − body size step)')
axes[0].set_ylabel('Count')
axes[0].set_title('Fit Delta Distribution')
axes[0].legend()

# 2. Body size
bc = Counter(body_idxs)
axes[1].bar([SIZE_LABELS[i] for i in sorted(bc)], [bc[i] for i in sorted(bc)],
            color='coral', edgecolor='white')
axes[1].set_xlabel('Body Size'); axes[1].set_ylabel('Count')
axes[1].set_title('Body Size Distribution')

# 3. Height
axes[2].hist(heights, bins=25, color='mediumseagreen', edgecolor='white')
axes[2].axvline(np.mean(heights), color='red', linestyle='--',
                label=f'Mean={np.mean(heights):.0f}cm')
axes[2].set_xlabel('Height (cm)'); axes[2].set_ylabel('Count')
axes[2].set_title('Height Distribution'); axes[2].legend()

# 4. Weight
axes[3].hist(weights, bins=25, color='mediumpurple', edgecolor='white')
axes[3].axvline(np.mean(weights), color='red', linestyle='--',
                label=f'Mean={np.mean(weights):.0f}kg')
axes[3].set_xlabel('Weight (kg)'); axes[3].set_ylabel('Count')
axes[3].set_title('Weight Distribution'); axes[3].legend()

plt.tight_layout()
plt.savefig('outputs/dataset_stats.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → outputs/dataset_stats.png')

In [ ]:
# ── Cell 5: MeasurementEncoder — Token Inspection ───────────────────────────
from fit_vton.models.measurement_encoder import MeasurementEncoder
from fit_vton.data.fit_dataset import MEASUREMENT_MEAN, MEASUREMENT_STD

encoder = MeasurementEncoder(embed_dim=768, hidden_dim=512, num_tokens=4, num_garment_sizes=6)
encoder.eval()
print(f'MeasurementEncoder  |  Parameters: {encoder.num_parameters:,}')

# ── Scenario: 170cm/65kg woman (body=M) with different garment sizes ─────
body_raw = torch.tensor([[170.0, 65.0, 90.0, 72.0, 97.0]])  # M-ish body
body_norm = (body_raw - MEASUREMENT_MEAN) / MEASUREMENT_STD

scenarios = [
    ('XS', -3), ('S', -2), ('M', 0), ('L', 1), ('XL', 2), ('XXL', 3)
]

token_norms, token_first_dims, cosine_to_perfect = [], [], []
perfect_tokens = None

for garment_size, delta in scenarios:
    g_idx = torch.tensor([SIZE_TO_IDX[garment_size]])
    d_t   = torch.tensor([[float(delta)]])
    with torch.no_grad():
        tokens = encoder(body_norm, g_idx, d_t)  # (1, 4, 768)
    norm = tokens.norm().item()
    token_norms.append(norm)
    token_first_dims.append(tokens[0, 0, :3].tolist())
    if delta == 0:
        perfect_tokens = tokens.clone()
    if perfect_tokens is not None:
        cs = F.cosine_similarity(
            tokens.flatten().unsqueeze(0),
            perfect_tokens.flatten().unsqueeze(0)
        ).item()
        cosine_to_perfect.append(cs)
    else:
        cosine_to_perfect.append(None)

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('MeasurementEncoder Output Analysis (Body=M, varying garment size)', fontsize=13, fontweight='bold')

labels = [f'{g}\n(Δ={d:+d})' for g, d in scenarios]
x = np.arange(len(scenarios))

bars = axes[0].bar(x, token_norms, color='steelblue', edgecolor='white')
bars[2].set_color('red')  # perfect fit
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_ylabel('Token L2 Norm')
axes[0].set_title('Token Magnitude per Fit Scenario')
axes[0].text(2, token_norms[2]*0.85, 'perfect\nfit', ha='center', color='white', fontsize=8, fontweight='bold')

valid_cs  = [c for c in cosine_to_perfect if c is not None]
valid_lbl = [labels[i] for i, c in enumerate(cosine_to_perfect) if c is not None]
axes[1].plot(valid_lbl, valid_cs, marker='o', color='coral', linewidth=2)
axes[1].axhline(1.0, color='red', linestyle='--', linewidth=1, label='Perfect fit (ref)')
axes[1].set_ylabel('Cosine Similarity to Perfect-Fit Token')
axes[1].set_title('Token Similarity vs Perfect Fit')
axes[1].set_ylim(0, 1.1); axes[1].legend()
for lbl, val in zip(valid_lbl, valid_cs):
    axes[1].annotate(f'{val:.3f}', (lbl, val), textcoords='offset points', xytext=(0,6), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/token_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → outputs/token_analysis.png')

In [ ]:
# ── Cell 6: Simulate Try-On Results ─────────────────────────────────────────
#
# Since IDM-VTON requires ~10 GB download, we simulate try-on results using
# geometry transforms that capture the expected visual behaviour:
#
#   Baseline (IDM-VTON, no fit conditioning): always renders garment at
#   nominal size regardless of body/garment size delta.
#
#   FIT-VTON (with adapter): scales garment region proportionally to
#   fit_delta — tight garments appear form-fitting, baggy ones drape wider.
#
# This is a visual demonstration of what the trained adapter should learn.

def load_pil(path):
    return Image.open(path).convert('RGB').resize((256, 256))

def simulate_baseline(person: Image.Image, garment: Image.Image) -> Image.Image:
    """IDM-VTON baseline: overlay garment at fixed scale (no fit conditioning)."""
    result = person.copy().convert('RGBA')
    g = garment.convert('RGBA').resize((110, 130))
    W, H = result.size
    # paste garment on body region
    result.paste(g, (W//2 - 55, H//4), g)
    out = result.convert('RGB')
    # add slight blur to indicate it's the baseline (no adaptation)
    out = out.filter(ImageFilter.GaussianBlur(radius=0.6))
    return out

def simulate_fitvton(person: Image.Image, garment: Image.Image, fit_delta: float) -> Image.Image:
    """FIT-VTON: scale garment width by fit_delta to simulate drape change."""
    result = person.copy().convert('RGBA')
    # fit_delta > 0 → garment wider than body → baggy (scale up)
    # fit_delta < 0 → garment tighter than body → fitted/tight (scale down)
    scale = 1.0 + fit_delta * 0.12          # ±12% per size step
    scale = max(0.5, min(1.8, scale))
    base_w, base_h = 110, 130
    new_w = int(base_w * scale)
    new_h = int(base_h * (1 + fit_delta * 0.04))
    g = garment.convert('RGBA').resize((new_w, new_h))
    W, H = result.size
    result.paste(g, (W//2 - new_w//2, H//4), g)
    return result.convert('RGB')

print('Simulation functions defined.')
print('simulate_baseline → fixed garment overlay (no fit info)')
print('simulate_fitvton  → fit_delta-scaled garment (adapter behaviour)')

In [ ]:
# ── Cell 7: Main Comparison Grid ────────────────────────────────────────────
# Rows = different fit_delta scenarios
# Cols = [Person | Garment | Ground Truth | IDM-VTON Baseline | FIT-VTON (ours)]

# Pick one test sample per fit_delta bucket
buckets = defaultdict(list)
for m in test_meta:
    buckets[m['fit_delta']].append(m)

selected_deltas = [-2, -1, 0, 1, 2]
selected_samples = []
for d in selected_deltas:
    pool = buckets.get(d, [])
    if pool:
        selected_samples.append(random.choice(pool))
    else:
        # fallback: find closest
        closest = min(test_meta, key=lambda m: abs(m['fit_delta'] - d))
        selected_samples.append(closest)

N = len(selected_samples)
col_titles = ['Person', 'Garment', 'Ground Truth', 'IDM-VTON\n(baseline)', 'FIT-VTON\n(ours)']
NCOLS = len(col_titles)

fig, axes = plt.subplots(N, NCOLS, figsize=(NCOLS * 2.8, N * 3.0))
fig.suptitle('FIT-VTON vs IDM-VTON Baseline — Simulated Comparison\n'
             'Rows = different fit scenarios | Body size fixed at M',
             fontsize=13, fontweight='bold', y=1.01)

# Column headers
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight='bold',
                         color='#e07b54' if 'baseline' in title.lower()
                               else '#4a90d9' if 'ours' in title.lower()
                               else 'black')

for i, meta in enumerate(selected_samples):
    sid = meta['sample_id']
    delta = meta['fit_delta']
    g_size = SIZE_LABELS[meta['garment_size_idx']]
    b_size = SIZE_LABELS[meta['body_size_idx']]

    person_path  = f"{DATASET_DIR}/test/person/{sid}.jpg"
    garment_path = f"{DATASET_DIR}/test/garment/{sid}.jpg"
    tryon_path   = f"{DATASET_DIR}/test/tryon/{sid}.jpg"

    person_img  = load_pil(person_path)
    garment_img = load_pil(garment_path)
    tryon_img   = load_pil(tryon_path)
    baseline_img = simulate_baseline(person_img, garment_img)
    fitvton_img  = simulate_fitvton(person_img, garment_img, float(delta))

    images = [person_img, garment_img, tryon_img, baseline_img, fitvton_img]

    for j, img in enumerate(images):
        ax = axes[i, j]
        ax.imshow(np.array(img))
        ax.axis('off')
        # Highlight comparison columns
        if j == 3:
            for spine in ax.spines.values():
                spine.set_edgecolor('#e07b54'); spine.set_linewidth(2)
                spine.set_visible(True)
        elif j == 4:
            for spine in ax.spines.values():
                spine.set_edgecolor('#4a90d9'); spine.set_linewidth(2)
                spine.set_visible(True)

    # Row label
    fit_label = {
        -2: 'Too tight\n(Δ=−2)',
        -1: 'Slightly tight\n(Δ=−1)',
         0: 'Perfect fit\n(Δ=0)',
         1: 'Slightly baggy\n(Δ=+1)',
         2: 'Very baggy\n(Δ=+2)',
    }.get(delta, f'Δ={delta:+d}')
    axes[i, 0].set_ylabel(
        f'{fit_label}\nBody {b_size} → Garment {g_size}',
        fontsize=8, rotation=0, ha='right', va='center', labelpad=80
    )

plt.tight_layout()
plt.savefig('outputs/comparison_grid.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → outputs/comparison_grid.png')

In [ ]:
# ── Cell 8: Simulated Metric Comparison ─────────────────────────────────────
#
# Simulated numbers reflect realistic expected gains based on the FIT paper
# and typical IP-Adapter fine-tuning results.
# Replace with real numbers from evaluate.py after training.

metrics = {
    'FID ↓':          {'IDM-VTON\n(baseline)': 9.156,  'FIT-VTON\n(ours)': 7.830},
    'LPIPS ↓':        {'IDM-VTON\n(baseline)': 0.102,  'FIT-VTON\n(ours)': 0.087},
    'SSIM ↑':         {'IDM-VTON\n(baseline)': 0.868,  'FIT-VTON\n(ours)': 0.881},
    'Fit Accuracy ↑': {'IDM-VTON\n(baseline)': 0.431,  'FIT-VTON\n(ours)': 0.712},
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Metric Comparison: IDM-VTON Baseline vs FIT-VTON (Ours)\n'
             '(simulated — run evaluate.py for real numbers)',
             fontsize=12, fontweight='bold')

for ax, (metric, vals) in zip(axes, metrics.items()):
    methods = list(vals.keys())
    values  = list(vals.values())
    bar_colors = [COLORS[m] for m in methods]
    bars = ax.bar(methods, values, color=bar_colors, edgecolor='white', width=0.5)

    lower_better = '↓' in metric
    best_idx = np.argmin(values) if lower_better else np.argmax(values)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2.5)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Delta annotation
    delta = values[1] - values[0]
    good = (delta < 0) if lower_better else (delta > 0)
    delta_str = f"{'+' if delta>0 else ''}{delta:.3f}"
    ax.set_title(f'{metric}\nΔ = {delta_str} {"✓" if good else "✗"}',
                 fontsize=10, color='green' if good else 'red')
    ax.set_xticklabels(methods, fontsize=8)
    ax.set_ylabel('Score')
    ymin = min(values) * 0.92
    ymax = max(values) * 1.08
    ax.set_ylim(ymin, ymax)

plt.tight_layout()
plt.savefig('outputs/metrics_comparison.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → outputs/metrics_comparison.png')

# Print table
print(f"\n{'Metric':<18} {'IDM-VTON':>12} {'FIT-VTON':>12} {'Delta':>10} {'Better?':>8}")
print('-' * 65)
for metric, vals in metrics.items():
    b, f = list(vals.values())
    d = f - b
    lower = '↓' in metric
    good = (d < 0) if lower else (d > 0)
    print(f"{metric:<18} {b:>12.3f} {f:>12.3f} {d:>+10.3f} {'✓ ours' if good else '✗':>8}")

In [ ]:
# ── Cell 9: Fit Accuracy vs Fit Delta Plot ───────────────────────────────────
# Shows how Fit Accuracy (our custom metric) varies per fit scenario.
# FIT-VTON should outperform baseline most at extreme deltas (very tight/baggy).

deltas = np.array([-3, -2, -1, 0, 1, 2, 3])

# Baseline: flat (no fit awareness) with slight noise
np.random.seed(42)
baseline_acc = 0.43 + np.random.randn(len(deltas)) * 0.02

# FIT-VTON: peaks at 0 (perfect fit easiest), degrades gracefully at extremes
fitvton_acc  = 0.85 * np.exp(-0.08 * deltas**2) + 0.15 + np.random.randn(len(deltas)) * 0.015
fitvton_acc  = np.clip(fitvton_acc, 0.3, 0.95)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Fit Accuracy by Fit Delta — FIT-VTON vs Baseline', fontsize=13, fontweight='bold')

# Line plot
ax = axes[0]
ax.plot(deltas, baseline_acc, 'o--', color='#e07b54', linewidth=2, markersize=7, label='IDM-VTON (baseline)')
ax.plot(deltas, fitvton_acc,  's-',  color='#4a90d9', linewidth=2, markersize=7, label='FIT-VTON (ours)')
ax.fill_between(deltas, baseline_acc, fitvton_acc, alpha=0.15, color='#4a90d9')
ax.set_xlabel('Fit Delta (garment − body size step)')
ax.set_ylabel('Fit Accuracy')
ax.set_title('Fit Accuracy per Scenario')
ax.legend(); ax.set_ylim(0.2, 1.0)
ax.axvline(0, color='gray', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xticks(deltas)
ax.set_xticklabels([f'Δ={d:+d}\n({"tight" if d<0 else "baggy" if d>0 else "perfect"})'
                    for d in deltas], fontsize=8)

# Gain bar plot
ax2 = axes[1]
gains = fitvton_acc - baseline_acc
bar_colors = ['#4a90d9' if g > 0 else '#e07b54' for g in gains]
ax2.bar(deltas, gains, color=bar_colors, edgecolor='white', width=0.6)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Fit Delta')
ax2.set_ylabel('Fit Accuracy Gain (FIT-VTON − baseline)')
ax2.set_title('Absolute Gain of FIT-VTON over Baseline')
ax2.set_xticks(deltas)
ax2.set_xticklabels([f'{d:+d}' for d in deltas])
for x, g in zip(deltas, gains):
    ax2.text(x, g + (0.005 if g >= 0 else -0.012), f'+{g:.2f}' if g >= 0 else f'{g:.2f}',
             ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/fit_accuracy_by_delta.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → outputs/fit_accuracy_by_delta.png')

In [ ]:
# ── Cell 10: Training Loss Curves (Simulated) ────────────────────────────────
# Simulates realistic training loss curves for the adapter.
# Replace with real wandb data after training.

np.random.seed(0)
steps = np.arange(0, 50001, 100)

def smooth_loss(start, end, n, noise=0.03):
    base = start * np.exp(-3.5 * np.linspace(0, 1, n)) + end
    return base + np.random.randn(n) * noise * base

train_loss    = smooth_loss(0.25, 0.04, len(steps), noise=0.08)
val_loss      = smooth_loss(0.27, 0.052, len(steps), noise=0.04)
baseline_loss = np.full(len(steps), 0.082) + np.random.randn(len(steps)) * 0.003  # flat

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Training Curves — FIT Adapter Fine-Tuning (Simulated)', fontsize=13, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(steps, train_loss, color='#4a90d9', linewidth=1.5, alpha=0.8, label='Train loss (adapter)')
ax.plot(steps, val_loss,   color='#4a90d9', linewidth=2,   linestyle='--', label='Val loss (adapter)')
ax.axhline(baseline_loss.mean(), color='#e07b54', linewidth=1.5, linestyle=':', label='IDM-VTON baseline loss')
ax.set_xlabel('Training Step')
ax.set_ylabel('Diffusion MSE Loss')
ax.set_title('Loss Curves')
ax.legend(); ax.set_xlim(0, 50000)

# Fit accuracy over training
ax2 = axes[1]
fit_acc_curve = 0.43 + 0.29 * (1 - np.exp(-4 * np.linspace(0, 1, len(steps)))) \
                + np.random.randn(len(steps)) * 0.008
ax2.plot(steps, fit_acc_curve, color='#4a90d9', linewidth=2, label='FIT-VTON fit accuracy')
ax2.axhline(0.43, color='#e07b54', linewidth=1.5, linestyle=':', label='Baseline fit accuracy')
ax2.set_xlabel('Training Step')
ax2.set_ylabel('Fit Accuracy')
ax2.set_title('Fit Accuracy During Training')
ax2.legend(); ax2.set_xlim(0, 50000)
ax2.set_ylim(0.3, 0.85)

plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved → outputs/training_curves.png')

In [ ]:
# ── Cell 11: Summary & Next Steps ────────────────────────────────────────────
from pathlib import Path

print('=' * 60)
print('FIT-VTON Colab Demo Complete!')
print('=' * 60)
print()
print('Outputs generated:')
for f in sorted(Path('outputs').glob('*.png')):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:<35} ({size_kb} KB)')
print()
print('What this notebook demonstrated:')
print('  1. Synthetic FIT dataset generation (realistic measurements)')
print('  2. MeasurementEncoder: different fit scenarios → different tokens')
print('  3. Simulated visual comparison: Baseline vs FIT-VTON')
print('  4. Metric comparison table (FID / LPIPS / SSIM / Fit Accuracy)')
print('  5. Fit Accuracy breakdown by fit delta')
print('  6. Simulated training curves')
print()
print('Next steps to get REAL results:')
print('  python scripts/download_base_model.py')
print('  python scripts/download_fit_dataset.py --mini')
print('  python train.py --config configs/train.yaml')
print('  python evaluate.py --checkpoint outputs/checkpoints/step_50000')
print()
print('GitHub: https://github.com/NatalieCarlebach1/fit-vton')